# 🧄 EfficientNetB4 + E-FSDA v8: Coordinate Attention
# 
# **Đề tài:** Phân loại tỏi (Garlic Classification) sử dụng EfficientNetB4 + E-FSDA
# 
# ## Đóng góp chính (Novel Contributions):
# 1. **Coordinate Attention** (CVPR 2021 adaptation) — thay thế spatial attention đơn giản  
#    bằng cơ chế encode vị trí (x, y) chính xác, giúp model biết defect/peeling ở đâu
# 2. **Frequency Channel Attention + Learnable Temperature** — mỗi channel có temperature riêng,  
#    cho phép attention sharpness tự điều chỉnh (sharp focus vs diffuse)
# 3. **Gated Fusion + Residual** — thay vì addition cố định, dùng learnable gate  
#    để tự cân bằng freq vs spatial, kèm residual connection ổn định training
# 
# ## Pipeline:
# ```
# EfficientNetB4 (pretrained) → E-FSDA v8 Block → GAP → Head → Softmax
#                                    ├── FreqChannelAttn (FFT + MLP + Temperature)
#                                    ├── CoordinateAttn (H-pool + W-pool + positional)
#                                    └── Gated Fusion (learnable α) + Residual
# ```
# 
# **Platform:** Kaggle GPU (P100/T4) | **Framework:** TensorFlow 2.x + Keras 3 | **Mixed Precision:** float16

In [ ]:
# ============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ============================================================================
# Tất cả thư viện cần thiết cho training pipeline.
# Notebook này chạy ĐỘC LẬP — không phụ thuộc file nào khác.
# ============================================================================

import os
import csv
import time
import random
import shutil
import glob
import gc
from collections import defaultdict
from types import SimpleNamespace

# --- Scientific Computing ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Deep Learning Framework ---
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Activation, Input, Layer, Reshape, Concatenate,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, CSVLogger, Callback,
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# --- Scikit-learn Metrics & Utilities ---
from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ============================================================================
# GPU CONFIGURATION + MIXED PRECISION
# ============================================================================
print("=" * 60)
print("  ENVIRONMENT SETUP")
print("=" * 60)
print(f"  TensorFlow version : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"  GPU(s) detected    : {len(gpus)} — {[g.name for g in gpus]}")
else:
    print("  ⚠️  No GPU detected — training will be slow!")

# Mixed precision: sử dụng float16 cho forward/backward, float32 cho loss/metrics
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"  Mixed Precision    : mixed_float16 (faster training, less memory)")
print(f"  XLA JIT            : Enabled")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION & HYPERPARAMETERS
# ============================================================================
# Tất cả hyperparameters tập trung tại đây để dễ chỉnh sửa.
# Khi chạy trên dataset khác, chỉ cần đổi DATA_DIR.
# ============================================================================

# --- Experiment Identification ---
STRATEGY_KEY   = "efsda_v8_coord_attention"
STRATEGY_LABEL = "EfficientNetB4 + E-FSDA v8 (Coordinate Attention + Freq Temperature)"

# --- Data Paths (CHỈNH ĐƯỜNG DẪN DATASET TẠI ĐÂY) ---
# Dataset-1 (2134 ảnh): "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
# Dataset-2 (2944 ảnh): "/kaggle/input/datasets/giaphuc/dataset-garlic-2944/dataset_final_2944"
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model Architecture ---
INPUT_SHAPE     = (380, 380, 3)       # EfficientNetB4 recommended input size
BATCH_SIZE      = 32                   # Giảm xuống 16 nếu OOM
EPOCHS          = 30                   # Max epochs (early stopping sẽ dừng sớm)
LR              = 1e-4                 # Initial learning rate (Adam)
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]     # Fine-tune top blocks của EfficientNetB4
DROPOUT_RATE    = 0.5                  # Dropout trước softmax
PATIENCE        = 12                   # Early stopping patience

# --- E-FSDA v8 Specific Hyperparameters ---
EFSDA_REDUCTION = 16                   # Channel reduction ratio cho attention MLPs

# --- Loss Function ---
FOCAL_GAMMA  = 2.0                     # Focal loss focusing parameter
CB_BETA      = 0.9999                  # Class-balanced effective number beta
ADAPTIVE_TAU = 0.3                     # EMA smoothing cho adaptive weight update

# --- Reproducibility ---
N_RUNS       = 3                       # Số lần chạy (statistical significance)
RANDOM_SEEDS = [42, 123, 456]          # Fixed seeds cho reproducibility
AUTOTUNE     = tf.data.AUTOTUNE        # tf.data pipeline optimization
tf.config.optimizer.set_jit(True)      # XLA JIT compilation

# --- Runtime Storage ---
all_runs_results = []                  # Accumulate results across runs

# --- Print Summary ---
print("=" * 60)
print("  EXPERIMENT CONFIGURATION")
print("=" * 60)
print(f"  Strategy    : {STRATEGY_LABEL}")
print(f"  Dataset     : {DATA_DIR.split('/')[-1]}")
print(f"  Input Shape : {INPUT_SHAPE}")
print(f"  Batch Size  : {BATCH_SIZE}")
print(f"  Epochs      : {EPOCHS} (patience={PATIENCE})")
print(f"  LR          : {LR} (ExponentialDecay)")
print(f"  Unfreeze    : blocks {UNFREEZE_BLOCKS}")
print(f"  Runs        : {N_RUNS} × seeds {RANDOM_SEEDS}")
print("-" * 60)
print(f"  [Novel 1] Coordinate Attention — positional encoding (H+W pooling)")
print(f"  [Novel 2] Frequency Channel Attention — learnable temperature σ(x/τ)")
print(f"  [Novel 3] Gated Fusion — α·freq + (1-α)·coord + residual")
print("=" * 60)

In [ ]:
# ========== 3. E-FSDA v8: COORDINATE ATTENTION + FREQUENCY ATTENTION ========== #

class FrequencyChannelAttentionV8(Layer):
    """Frequency Channel Attention with Learnable Temperature.
    
    Uses FFT magnitude spectrum + MLP + temperature-scaled sigmoid.
    """
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 8)
        
        self.fc1 = Dense(r, activation='relu', use_bias=False, dtype='float32',
                         name=f'{self.name}_fc1')
        self.fc2 = Dense(C, use_bias=False, dtype='float32',
                         name=f'{self.name}_fc2')
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        
        # Learnable temperature per channel
        self.temperature = self.add_weight(
            name='freq_temperature', shape=(1, C),
            initializer=tf.keras.initializers.Ones(),
            trainable=True, constraint=tf.keras.constraints.NonNeg())
        
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        
        # FFT -> magnitude spectrum -> GAP
        x_t = tf.transpose(x_f32, [0, 3, 1, 2])  # (B, C, H, W)
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft = tf.signal.fft2d(x_complex)
        mag = tf.math.log1p(tf.abs(x_fft))  # (B, C, H, W)
        
        # Global descriptor from frequency domain
        freq_desc = tf.reduce_mean(mag, axis=[2, 3])  # (B, C)
        
        # MLP
        freq_desc = self.fc1(freq_desc)
        freq_desc = self.fc2(freq_desc)
        
        # Temperature-scaled sigmoid
        temperature = tf.nn.softplus(tf.cast(self.temperature, tf.float32)) + 1e-6
        attn = tf.nn.sigmoid(freq_desc / temperature)  # (B, C)
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        
        out = x_f32 * attn
        return tf.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction})
        return cfg


class CoordinateAttention(Layer):
    """Coordinate Attention (CVPR 2021) adapted for feature maps.
    
    Encodes long-range spatial dependencies with precise positional info.
    Decomposes channel attention into two 1D feature encoding processes
    that aggregate features along H and W directions respectively.
    
    Novel adaptation: Added learnable spatial temperature.
    """
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        H = input_shape[1] or 12
        W = input_shape[2] or 12
        r = max(C // self.reduction, 8)
        
        # Shared transform (after concatenating H-pooled and W-pooled)
        self.conv_reduce = Conv2D(r, 1, use_bias=False, dtype='float32',
                                  name=f'{self.name}_conv_reduce')
        self.bn_reduce = BatchNormalization(dtype='float32',
                                           name=f'{self.name}_bn_reduce')
        
        # Separate transforms for H and W
        self.conv_h = Conv2D(C, 1, use_bias=False, dtype='float32',
                             name=f'{self.name}_conv_h')
        self.conv_w = Conv2D(C, 1, use_bias=False, dtype='float32',
                             name=f'{self.name}_conv_w')
        
        # Spatial temperature
        self.spatial_temperature = self.add_weight(
            name='coord_temperature', shape=(1,),
            initializer=tf.keras.initializers.Ones(),
            trainable=True, constraint=tf.keras.constraints.NonNeg())
        
        # Build sub-layers
        self.conv_reduce.build((None, None, None, C))
        self.bn_reduce.build((None, None, None, r))
        self.conv_h.build((None, None, None, r))
        self.conv_w.build((None, None, None, r))
        
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        B = tf.shape(x_f32)[0]
        H = tf.shape(x_f32)[1]
        W = tf.shape(x_f32)[2]
        C = tf.shape(x_f32)[3]
        
        # Pool along W axis -> (B, H, 1, C): captures vertical position
        x_h = tf.reduce_mean(x_f32, axis=2, keepdims=True)  # (B, H, 1, C)
        
        # Pool along H axis -> (B, 1, W, C): captures horizontal position  
        x_w = tf.reduce_mean(x_f32, axis=1, keepdims=True)  # (B, 1, W, C)
        
        # Transpose x_w to (B, W, 1, C) for concatenation with x_h along spatial dim
        x_w_t = tf.transpose(x_w, [0, 2, 1, 3])  # (B, W, 1, C)
        
        # Concatenate along height dimension: (B, H+W, 1, C)
        y = tf.concat([x_h, x_w_t], axis=1)  # (B, H+W, 1, C)
        
        # Shared 1x1 conv + BN + ReLU
        y = self.conv_reduce(y)
        y = self.bn_reduce(y, training=training)
        y = tf.nn.relu(y)  # (B, H+W, 1, r)
        
        # Split back
        x_h_out, x_w_out = tf.split(y, [H, W], axis=1)
        # x_h_out: (B, H, 1, r), x_w_out: (B, W, 1, r)
        
        # Separate 1x1 conv for each
        x_h_out = self.conv_h(x_h_out)  # (B, H, 1, C)
        x_w_out = self.conv_w(x_w_out)  # (B, W, 1, C)
        
        # Temperature-scaled sigmoid
        sp_temp = tf.nn.softplus(tf.cast(self.spatial_temperature, tf.float32)) + 1e-6
        a_h = tf.nn.sigmoid(x_h_out / sp_temp)  # (B, H, 1, C)
        
        # Transpose x_w_out back: (B, W, 1, C) -> (B, 1, W, C)
        a_w = tf.nn.sigmoid(tf.transpose(x_w_out, [0, 2, 1, 3]) / sp_temp)  # (B, 1, W, C)
        
        # Apply coordinate attention
        out = x_f32 * a_h * a_w  # Broadcasting: (B,H,W,C) * (B,H,1,C) * (B,1,W,C)
        
        return tf.cast(out, x.dtype), a_h * a_w  # Return attention map for visualization

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction})
        return cfg


class EFSDAv8Block(Layer):
    """E-FSDA v8: Frequency Channel Attention + Coordinate Attention + Gated Fusion.
    
    Novel contributions:
    1. Coordinate Attention replaces simple spatial attention (positional encoding)
    2. Frequency attention with learnable temperature
    3. Learnable gated fusion (instead of simple addition)
    4. Residual connection for training stability
    """
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        
        self.freq_attn = FrequencyChannelAttentionV8(
            reduction=self.reduction, name=f'{self.name}_freq_attn')
        
        self.coord_attn = CoordinateAttention(
            reduction=self.reduction, name=f'{self.name}_coord_attn')
        
        # Gated fusion: learns balance between freq and coord branches
        self.gate_fc1 = Dense(C // 4, activation='relu', use_bias=False, dtype='float32',
                              name=f'{self.name}_gate_fc1')
        self.gate_fc2 = Dense(C, activation='sigmoid', use_bias=False, dtype='float32',
                              name=f'{self.name}_gate_fc2')
        
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        
        # Build sub-layers
        self.freq_attn.build(input_shape)
        self.coord_attn.build(input_shape)
        self.gate_fc1.build((None, C))
        self.gate_fc2.build((None, C // 4))
        self.bn.build(input_shape)
        
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        x_f32 = tf.cast(x, tf.float32)
        
        # Frequency branch: "what features matter" (channel-wise)
        freq_out = tf.cast(self.freq_attn(x, training=training), tf.float32)
        
        # Coordinate Attention branch: "where to look" (with positional info)
        coord_out, coord_attn_map = self.coord_attn(x, training=training)
        coord_out = tf.cast(coord_out, tf.float32)
        
        # Gated fusion: learn to balance freq vs coord
        gap = tf.reduce_mean(x_f32, axis=[1, 2])  # (B, C)
        gate = self.gate_fc1(gap)
        gate = self.gate_fc2(gate)  # (B, C) in [0, 1]
        gate = tf.reshape(gate, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        
        # Fuse: gate * freq + (1-gate) * coord + residual
        fused = gate * freq_out + (1.0 - gate) * coord_out + x_f32
        fused = self.bn(fused, training=training)
        
        return tf.cast(fused, input_dtype), coord_attn_map

    def compute_output_spec(self, x, training=False):
        import keras
        sp_shape = tuple(x.shape[:-1]) + (x.shape[-1],)
        return (
            keras.KerasTensor(x.shape, dtype=x.dtype),
            keras.KerasTensor(sp_shape, dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction})
        return cfg


print("E-FSDA v8 (Coordinate Attention) defined.")
print("  Novel 1: Coordinate Attention with positional encoding (CVPR 2021 adaptation)")
print("  Novel 2: Frequency Channel Attention with learnable temperature")
print("  Novel 3: Gated Fusion + Residual Connection")

In [ ]:
# ========== 4. ADAPTIVE CLASS-BALANCED FOCAL LOSS ========== #

class AdaptiveClassBalancedFocalLoss(tf.keras.losses.Loss):
    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.beta = beta
        self.num_classes = num_classes
        self._samples_per_class = list(samples_per_class)
        n = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * len(samples_per_class)
        self.static_weights = tf.constant(weights, dtype=tf.float32)
        self.adaptive_factor = tf.Variable(
            tf.ones([num_classes], dtype=tf.float32),
            trainable=False, name='adaptive_cb_factor')

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        combined_weights = self.static_weights * self.adaptive_factor
        combined_weights = combined_weights / tf.reduce_mean(combined_weights)
        sample_w = tf.reduce_sum(y_true * combined_weights, axis=-1)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'samples_per_class': self._samples_per_class,
                    'num_classes': self.num_classes,
                    'gamma': self.gamma, 'beta': self.beta})
        return cfg


class AdaptiveWeightCallback(Callback):
    def __init__(self, loss_fn, val_ds, num_classes, class_names, tau=0.3, **kwargs):
        super().__init__(**kwargs)
        self.loss_fn = loss_fn
        self.val_ds = val_ds
        self.num_classes = num_classes
        self.class_names = class_names
        self.tau = tau
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in self.val_ds])
        per_class_recall = np.zeros(self.num_classes)
        for c in range(self.num_classes):
            mask = y_true == c
            per_class_recall[c] = (y_pred[mask] == c).mean() if mask.sum() > 0 else 1.0
        epsilon = 0.1
        adaptation_target = (1.0 - per_class_recall) + epsilon
        current_factor = self.loss_fn.adaptive_factor.numpy()
        new_factor = (1.0 - self.tau) * current_factor + self.tau * adaptation_target
        new_factor = new_factor / new_factor.mean()
        self.loss_fn.adaptive_factor.assign(new_factor.astype(np.float32))
        self.history.append({'epoch': epoch+1,
                             'per_class_recall': per_class_recall.copy(),
                             'adaptive_factor': new_factor.copy()})

print("Adaptive CB Focal Loss + Callback defined.")

In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    base.trainable = False
    for layer in base.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Backbone: {trainable}/{len(base.layers)} layers trainable")


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        return img, tf.one_hot(label, depth=num_classes)

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    cw = class_weight.compute_class_weight('balanced', classes=np.unique(train_lbl), y=train_lbl)
    meta = SimpleNamespace(
        class_names=class_names, num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        class_weight_dict=dict(enumerate(cw)),
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    return train_ds, val_ds, test_ds, meta


print("Helpers defined.")

In [ ]:
# ========== 6. MODEL BUILDER ========== #

CUSTOM_OBJECTS = {
    'FrequencyChannelAttentionV8': FrequencyChannelAttentionV8,
    'CoordinateAttention': CoordinateAttention,
    'EFSDAv8Block': EFSDAv8Block,
    'AdaptiveClassBalancedFocalLoss': AdaptiveClassBalancedFocalLoss,
}


def build_efsda_v8_model(input_shape, num_classes, steps_per_epoch, samples_per_class):
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    feat_map = base.output

    attended, coord_attn_map = EFSDAv8Block(
        reduction=EFSDA_REDUCTION,
        name='efsda_v8',
    )(feat_map)

    x = GlobalAveragePooling2D(name='gap')(attended)
    x = BatchNormalization(name='head_bn')(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-5), name='head_dense')(x)
    x = Dropout(DROPOUT_RATE, name='head_dropout')(x)
    out = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out, name='EfficientNetB4_EFSDAv8')

    loss_fn = AdaptiveClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes, gamma=FOCAL_GAMMA, beta=CB_BETA)

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
                  loss=loss_fn, metrics=['accuracy'])
    return model, loss_fn


print("Model builder defined.")

In [ ]:
# ========== 7. MULTI-RUN TRAINING ========== #

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    samples_per_class = np.array([
        len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', cn))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for cn in meta.class_names
    ])

    model, loss_fn = build_efsda_v8_model(
        INPUT_SHAPE, meta.num_classes, steps_per_epoch, samples_per_class)

    if run_idx == 0:
        model.summary(print_fn=lambda x: print(x) if 'efsda' in x.lower() or 'Total' in x or 'Trainable' in x else None)

    adaptive_cb = AdaptiveWeightCallback(
        loss_fn=loss_fn, val_ds=val_ds,
        num_classes=meta.num_classes, class_names=meta.class_names, tau=ADAPTIVE_TAU)

    callbacks = [
        adaptive_cb,
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv')),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks)

    # Evaluate
    best_model = load_model(os.path.join(RESULT_DIR, 'best_model.keras'),
                            custom_objects=CUSTOM_OBJECTS)
    pred_probs = best_model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=meta.class_names, output_dict=True, digits=4)
    test_acc = np.mean(y_pred_run == y_true_run)

    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=meta.class_names, digits=4))

    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=meta.class_names,
                yticklabels=meta.class_names, cmap='Blues', ax=ax)
    ax.set_title(f'CM \u2014 Run {run_idx+1}'); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300); plt.close()

    all_runs_results.append({
        'run': run_idx+1, 'seed': seed,
        'accuracy': test_acc,
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'per_class_metrics': {c: report[c] for c in meta.class_names},
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true_run, 'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'class_names': meta.class_names,
        'test_filenames': meta.test_filenames,
        'n_train': meta.n_train, 'n_val': meta.n_val, 'n_test': meta.n_test,
        'adaptive_history': adaptive_cb.history,
    })

    print(f"  Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}  R={report['weighted avg']['recall']:.4f}  F1={report['weighted avg']['f1-score']:.4f}")
    tf.keras.backend.clear_session()

print("\n" + "="*70 + f"\n ALL {N_RUNS} RUNS COMPLETED\n" + "="*70)

In [ ]:
# ========== 8. RESULTS AGGREGATION ========== #

accuracies  = [r['accuracy'] for r in all_runs_results]
precisions  = [r['precision'] for r in all_runs_results]
recalls     = [r['recall'] for r in all_runs_results]
f1_scores   = [r['f1_score'] for r in all_runs_results]

print(f"\n{'='*60}")
print(f"  {STRATEGY_LABEL}")
print(f"{'='*60}")
print(f"  Accuracy  : {np.mean(accuracies):.4f} \u00b1 {np.std(accuracies):.4f}")
print(f"  Precision : {np.mean(precisions):.4f} \u00b1 {np.std(precisions):.4f}")
print(f"  Recall    : {np.mean(recalls):.4f} \u00b1 {np.std(recalls):.4f}")
print(f"  F1-Score  : {np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}")
print(f"  Per run acc: {[f'{a:.4f}' for a in accuracies]}")

for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    print(f"  Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

class_names = all_runs_results[0]['class_names']
print("\nPER-CLASS METRICS:")
print("-" * 70)
for cn in class_names:
    p_vals = [r['per_class_metrics'][cn]['precision'] for r in all_runs_results]
    r_vals = [r['per_class_metrics'][cn]['recall'] for r in all_runs_results]
    f_vals = [r['per_class_metrics'][cn]['f1-score'] for r in all_runs_results]
    print(f"  {cn:<30} P={np.mean(p_vals):.4f}\u00b1{np.std(p_vals):.4f}  "
          f"R={np.mean(r_vals):.4f}\u00b1{np.std(r_vals):.4f}  "
          f"F1={np.mean(f_vals):.4f}\u00b1{np.std(f_vals):.4f}")

# Save summary
summary_df = pd.DataFrame([{
    'strategy': STRATEGY_KEY, 'run': r['run'], 'seed': r['seed'],
    'accuracy': r['accuracy'], 'precision': r['precision'],
    'recall': r['recall'], 'f1_score': r['f1_score'],
} for r in all_runs_results])
summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'summary.csv'), index=False)

# Zip
zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
print(f"\n\u2705 Archived \u2192 {zip_path}.zip")